# EDA

In [70]:
import pandas as pd
import numpy as np

In [ ]:
# Load the preprocessed order-level dataset
df = pd.read_csv(
    "../01_data/DataCo_order_level.csv",
    parse_dates=[
        "order date (DateOrders)",
        "shipping date (DateOrders)",
    ],
)

# Confirm the dimensions of the order-level dataset
df.shape

(65752, 146)

In [57]:
# Verify that the dataset contains one record per order
assert df["Order Id"].is_unique

# Shipping

In [75]:
customer_orders = (
    df.groupby("Customer Id")
    .agg(
        orders=("Order Id", "nunique"),
        first_order=("order date (DateOrders)", "min"),
        last_order=("order date (DateOrders)", "max"),
    )
)

customer_orders["customer_span_days"] = (
    customer_orders["last_order"]
    - customer_orders["first_order"]
).dt.days

customer_orders[
    ["orders", "customer_span_days"]
].describe()

,orders,customer_span_days
count,20652.000000,20652.000000
mean,3.183808,352.583236
std,2.430699,353.388833
min,1.000000,0.000000
25%,1.000000,0.000000
50%,3.000000,305.000000
75%,5.000000,699.000000
max,15.000000,1003.000000


In [76]:
customer_history = df[
    ["Customer Id", "Order Id", "order date (DateOrders)"]
].sort_values(
    ["Customer Id", "order date (DateOrders)"]
)

customer_history["days_since_prior_order"] = (
    customer_history.groupby("Customer Id")[
        "order date (DateOrders)"
    ].diff().dt.total_seconds() / 86400
)

customer_history["days_since_prior_order"].describe()

count    45100.000000
mean       161.584009
std        149.663432
min          0.014583
25%         49.092361
50%        116.562500
75%        230.409722
max        989.431250
Name: days_since_prior_order, dtype: float64

In [77]:
customer_history[
    "days_since_prior_order"
].value_counts().head(20)

days_since_prior_order
18.247222    11
18.962500    10
9.138194     10
86.155556    10
29.720833     9
67.207639     9
6.963194      9
7.722222      8
2.992361      8
45.865972     8
4.934028      8
91.133333     8
8.919444      8
1.474306      8
35.370139     8
16.072222     8
13.984722     8
2.846528      8
24.013194     8
58.770139     8
Name: count, dtype: int64

In [59]:
pd.crosstab(
    df["Shipping Mode"],
    df["Days for shipment (scheduled)"],
    normalize="index"
)

Days for shipment (scheduled),0,1,2,4
Shipping Mode,,,,
First Class,0.0,1.0,0.0,0.0
Same Day,1.0,0.0,0.0,0.0
Second Class,0.0,0.0,1.0,0.0
Standard Class,0.0,0.0,0.0,1.0


In [60]:
delivered_df = df[
    df["Delivery Status"] != "Shipping canceled"
].copy()

pd.crosstab(
    delivered_df["Days for shipment (scheduled)"],
    delivered_df["Days for shipping (real)"],
    normalize="index",
).round(4) * 100

Days for shipping (real),0,1,2,3,4,5,6
Days for shipment (scheduled),,,,,,,
0,51.63,48.37,0.00,0.00,0.00,0.00,0.00
1,0.00,0.00,100.00,0.00,0.00,0.00,0.00
2,0.00,0.00,20.01,19.76,20.18,20.01,20.04
4,0.00,0.00,20.22,19.98,19.96,19.82,20.03


In [61]:
delivered_df["Order Year"] = delivered_df[
    "order date (DateOrders)"
].dt.year

pd.crosstab(
    [
        delivered_df["Order Year"],
        delivered_df["Days for shipment (scheduled)"],
    ],
    delivered_df["Days for shipping (real)"],
    normalize="index",
).round(4) * 100

Days for shipping (real)                      0      1       2      3      4  \
Order Year Days for shipment (scheduled)                                       
2015       0                              53.00  47.00    0.00   0.00   0.00   
           1                               0.00   0.00  100.00   0.00   0.00   
           2                               0.00   0.00   19.93  19.88  19.83   
           4                               0.00   0.00   20.26  20.30  19.70   
2016       0                              54.79  45.21    0.00   0.00   0.00   
           1                               0.00   0.00  100.00   0.00   0.00   
           2                               0.00   0.00   19.86  19.86  20.55   
           4                               0.00   0.00   19.94  19.71  19.98   
2017       0                              46.98  53.02    0.00   0.00   0.00   
           1                               0.00   0.00  100.00   0.00   0.00   
           2                               0.00   0.00   20.32  19.58  20.00   
           4                               0.00   0.00   20.47  19.95  20.22   
2018       0                              55.36  44.64    0.00   0.00   0.00   
           1                               0.00   0.00  100.00   0.00   0.00   
           2                               0.00   0.00   19.29  19.53  21.65   
           4                               0.00   0.00   20.10  19.59  19.51   

Days for shipping (real)                      5      6  
Order Year Days for shipment (scheduled)                
2015       0                               0.00   0.00  
           1                               0.00   0.00  
           2                              20.04  20.32  
           4                              19.70  20.05  
2016       0                               0.00   0.00  
           1                               0.00   0.00  
           2                              20.09  19.65  
           4                              20.06  20.31  
2017       0                               0.00   0.00  
           1                               0.00   0.00  
           2                              19.80  20.30  
           4                              19.72  19.64  
2018       0                               0.00   0.00  
           1                               0.00   0.00  
           2                              20.94  18.59  
           4                              19.68  21.12

The highly regular shipping-duration distributions persist independently across years. In particular, orders scheduled for two or four days remain distributed approximately evenly across two through six actual shipping days in each full year. Therefore, the pooled pattern does not appear to result from aggregating different historical shipping regimes.

In [62]:
# Compare order status with recorded delivery status
pd.crosstab(
    df["Order Status"],
    df["Delivery Status"],
    normalize="index",
).mul(100).round(2)

Delivery Status,Advance shipping,Late delivery,Shipping canceled,Shipping on time
Order Status,,,,
CANCELED,0.00,0.00,100.0,0.00
CLOSED,24.51,56.73,0.0,18.76
COMPLETE,23.83,57.66,0.0,18.52
ON_HOLD,23.68,56.43,0.0,19.90
PAYMENT_REVIEW,22.59,57.53,0.0,19.89
PENDING,24.55,57.57,0.0,17.88
PENDING_PAYMENT,23.79,57.41,0.0,18.81
PROCESSING,24.57,56.88,0.0,18.55
SUSPECTED_FRAUD,0.00,0.00,100.0,0.00


In [63]:
# Compare late-delivery rates across order statuses
pd.crosstab(
    df["Order Status"],
    df["Late_delivery_risk"],
    normalize="index",
).round(4) * 100

Late_delivery_risk,0,1
Order Status,,
CANCELED,100.00,0.00
CLOSED,43.27,56.73
COMPLETE,42.34,57.66
ON_HOLD,43.57,56.43
PAYMENT_REVIEW,42.47,57.53
PENDING,42.43,57.57
PENDING_PAYMENT,42.59,57.41
PROCESSING,43.12,56.88
SUSPECTED_FRAUD,100.00,0.00


In [64]:
df.groupby("Order Status").agg(
    orders=("Order Id", "size"),
    shipping_date_present=("shipping date (DateOrders)", "count"),
    actual_days_present=("Days for shipping (real)", "count"),
    scheduled_days_present=("Days for shipment (scheduled)", "count"),
)

,orders,shipping_date_present,actual_days_present,scheduled_days_present
Order Status,,,,
CANCELED,1367,1367,1367,1367
CLOSED,7249,7249,7249,7249
COMPLETE,21716,21716,21716,21716
ON_HOLD,3624,3624,3624,3624
PAYMENT_REVIEW,704,704,704,704
PENDING,7321,7321,7321,7321
PENDING_PAYMENT,14382,14382,14382,14382
PROCESSING,7901,7901,7901,7901
SUSPECTED_FRAUD,1488,1488,1488,1488


Order and shipping status validation: All orders contain a recorded shipping date and actual and scheduled shipping durations, including orders whose Order Status is CANCELED or SUSPECTED_FRAUD and whose Delivery Status is Shipping canceled. In addition, nonterminal order statuses such as PENDING_PAYMENT, PENDING, and PROCESSING contain realized shipping outcomes. These relationships indicate that the precise temporal meaning of the status and shipping fields cannot be established from the dataset alone. Therefore, Order Status is not used as an order-time predictor, and orders with Delivery Status = "Shipping canceled" are excluded from delivery-performance analysis.

In [65]:
df.groupby("Delivery Status")[
    "Days for shipping (real)"
].describe()

,count,mean,std,min,25%,50%,75%,max
Delivery Status,,,,,,,,
Advance shipping,15127.0,2.496926,0.500007,2.0,2.0,2.0,3.0,3.0
Late delivery,36048.0,4.092266,1.708637,1.0,2.0,5.0,6.0,6.0
Shipping canceled,2855.0,3.487215,1.636060,0.0,2.0,3.0,5.0,6.0
Shipping on time,11722.0,2.981232,1.483345,0.0,2.0,4.0,4.0,4.0


In [66]:
delivered_df["Schedule Variance"] = (
    delivered_df["Days for shipping (real)"]
    - delivered_df["Days for shipment (scheduled)"]
)

delivered_df["Schedule Variance"].value_counts(normalize=True).sort_index()

Schedule Variance
-2    0.120991
-1    0.119513
 0    0.186368
 1    0.335946
 2    0.159149
 3    0.038984
 4    0.039048
Name: proportion, dtype: float64

# Profitability

In [78]:
df[
    [
        "Total Order Profit",
        "Order Profit Margin",
        "Net Order Sales",
        "Total Order Discount",
    ]
].describe()

,Total Order Profit,Order Profit Margin,Net Order Sales,Total Order Discount
count,54814.000000,54814.000000,65752.000000,65752.000000
mean,63.820121,0.321134,502.713256,56.734067
std,187.240073,2.709655,320.948506,46.584295
min,-1845.090000,-141.711600,7.490000,0.000000
25%,11.330000,0.021200,244.780000,20.000000
50%,80.265000,0.166900,455.950000,47.390000
75%,167.030000,0.430975,721.190000,82.390000
max,632.210000,54.622000,2768.410000,681.500000


In [79]:
(df["Total Order Profit"] <= 0).value_counts(normalize=True)

Total Order Profit
False    0.819564
True     0.180436
Name: proportion, dtype: float64

In [80]:
df[
    [
        "Total Order Profit",
        "Order Profit Margin",
        "Net Order Sales",
        "Total Order Discount",
    ]
].corr().round(3)

,Total Order Profit,Order Profit Margin,Net Order Sales,Total Order Discount
Total Order Profit,1.000,0.370,-0.000,0.002
Order Profit Margin,0.370,1.000,-0.112,-0.085
Net Order Sales,-0.000,-0.112,1.000,0.710
Total Order Discount,0.002,-0.085,0.710,1.000


In [81]:
pd.crosstab(
    df["Order Status"],
    df["Total Order Profit"].isna(),
    normalize="index",
).mul(100).round(2)

Total Order Profit,False,True
Order Status,,
CANCELED,83.76,16.24
CLOSED,83.50,16.50
COMPLETE,83.59,16.41
ON_HOLD,83.06,16.94
PAYMENT_REVIEW,83.52,16.48
PENDING,82.72,17.28
PENDING_PAYMENT,83.50,16.50
PROCESSING,83.00,17.00
SUSPECTED_FRAUD,83.53,16.47


In [83]:
pd.crosstab(
    df["Delivery Status"],
    df["Total Order Profit"].isna(),
    normalize="index",
).mul(100).round(2)

Total Order Profit,False,True
Delivery Status,,
Advance shipping,83.57,16.43
Late delivery,83.32,16.68
Shipping canceled,83.64,16.36
Shipping on time,83.16,16.84


# Customer

In [85]:
cutoff = pd.Timestamp("2017-01-01")
horizon_end = pd.Timestamp("2018-01-01")

customer_history = (
    df.loc[
        df["order date (DateOrders)"] < cutoff,
        ["Customer Id", "Order Id", "order date (DateOrders)"],
    ]
    .groupby("Customer Id")
    .agg(
        prior_orders=("Order Id", "nunique"),
        first_order=("order date (DateOrders)", "min"),
        last_order=("order date (DateOrders)", "max"),
    )
)

future_customers = df.loc[
    (df["order date (DateOrders)"] >= cutoff)
    & (df["order date (DateOrders)"] < horizon_end),
    "Customer Id",
].unique()

customer_history["repurchased_12m"] = (
    customer_history.index.isin(future_customers).astype(int)
)

customer_history[
    ["prior_orders", "repurchased_12m"]
].describe()

,prior_orders,repurchased_12m
count,12026.000000,12026.000000
mean,3.472726,0.719524
std,1.753442,0.449250
min,1.000000,0.000000
25%,2.000000,0.000000
50%,3.000000,1.000000
75%,5.000000,1.000000
max,12.000000,1.000000


In [86]:
customer_history["repurchased_12m"].value_counts(
    normalize=True
).round(3)

repurchased_12m
1    0.72
0    0.28
Name: proportion, dtype: float64

In [87]:
customer_history["prior_orders"].value_counts().sort_index()

prior_orders
1     1456
2     2492
3     2737
4     2277
5     1484
6      890
7      411
8      179
9       71
10      24
11       4
12       1
Name: count, dtype: int64

In [88]:
customer_history["recency_days"] = (
    cutoff - customer_history["last_order"]
).dt.days

customer_history.groupby("repurchased_12m").agg(
    customers=("prior_orders", "size"),
    avg_prior_orders=("prior_orders", "mean"),
    median_prior_orders=("prior_orders", "median"),
    avg_recency_days=("recency_days", "mean"),
    median_recency_days=("recency_days", "median"),
).round(1)

,customers,avg_prior_orders,median_prior_orders,avg_recency_days,median_recency_days
repurchased_12m,,,,,
0,3373,3.5,3.0,190.5,142.0
1,8653,3.5,3.0,188.3,140.0


In [89]:
customer_history["recency_group"] = pd.qcut(
    customer_history["recency_days"],
    q=5,
    duplicates="drop",
)

customer_history.groupby(
    "recency_group",
    observed=True,
)["repurchased_12m"].agg(["count", "mean"]).round(3)

,count,mean
recency_group,,
"(-0.001, 46.0]",2406,0.708
"(46.0, 105.0]",2410,0.737
"(105.0, 185.0]",2405,0.719
"(185.0, 317.0]",2402,0.719
"(317.0, 730.0]",2403,0.715


In [90]:
customer_history.groupby(
    "prior_orders"
)["repurchased_12m"].agg(["count", "mean"]).round(3)

,count,mean
prior_orders,,
1,1456,0.720
2,2492,0.706
3,2737,0.726
4,2277,0.719
5,1484,0.729
6,890,0.710
7,411,0.740
8,179,0.721
9,71,0.704
